In [1]:
%pip install \
    "qiskit==2.2.3" \
    "qiskit-machine-learning==0.9.0" \
    "qiskit-optimization==0.7.0" \
    "qiskit-algorithms==0.4.0" \
    "packaging>=24.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 11.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.7 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=f83cc776bd5f4c507b5d3285564e56b0b1154d8511ff75d842b77e453eb9b112
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [14]:
%pip install "pillow>=10" "matplotlib>=3.9"

In [20]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('torch') is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', 'torch==2.5.1',
        '--index-url', 'https://download.pytorch.org/whl/cpu',
    ])

import torch
from PIL import Image
import matplotlib

print('PyTorch:', torch.__version__)
print('이미지·그래프 패키지:', Image.__version__, matplotlib.__version__)
probe = torch.tensor([1.0, 2.0], device='cpu')
assert torch.equal(probe * 2, torch.tensor([2.0, 4.0]))
print('CPU 텐서 연산 확인 완료')

PyTorch: 2.14.1+cpu
이미지·그래프 패키지: 11.3.0 3.10.0
CPU 텐서 연산 확인 완료


In [24]:
import sys
from importlib.metadata import version, requires
from packaging.requirements import Requirement
from packaging.markers import default_environment

import numpy as np
import scipy
import sklearn
import qiskit
import qiskit_algorithms
import qiskit_machine_learning
import qiskit_optimization
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

expected = {
    'qiskit': '2.2.3',
    'qiskit-machine-learning': '0.9.0',
    'qiskit-optimization': '0.7.0',
    'qiskit-algorithms': '0.4.0',
}
print('Python:', sys.version.split()[0])
environment = default_environment()
environment['extra'] = ''
for package, target in expected.items():
    installed = version(package)
    print(package, installed)
    assert installed == target, f'{package} 설치 셀을 다시 실행하세요.'
    for specification in requires(package) or []:
        requirement = Requirement(specification)
        if requirement.marker and not requirement.marker.evaluate(environment):
            continue
        actual = version(requirement.name)
        assert actual in requirement.specifier, f'의존성 확인: {specification}, 설치 {actual}'
for package in ('numpy', 'scipy', 'scikit-learn'):
    print(package, version(package))
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
probability = Statevector.from_instruction(qc).probabilities()
assert np.allclose(probability, [0.5, 0, 0, 0.5])
print('회로 확률:', probability)
print('패키지 의존성·import·회로 연산 확인 완료')

Python: 3.13.18
qiskit 2.2.3
qiskit-machine-learning 0.9.0
qiskit-optimization 0.7.0
qiskit-algorithms 0.4.0
numpy 2.4.6
scipy 1.16.3
scikit-learn 1.6.1
회로 확률: [0.5 0.  0.  0.5]
패키지 의존성·import·회로 연산 확인 완료


In [25]:
import numpy as np

logits = np.log(np.array([[0.70, 0.20, 0.10], [0.40, 0.35, 0.25]]))
shifted = logits - logits.max(axis=1, keepdims=True)
probability = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
threshold = 0.6
confidence = probability.max(axis=1)
base = probability.argmax(axis=1)
route = confidence < threshold
qml_prediction = np.array([2])
final = base.copy()
final[route] = qml_prediction
print('확률:', probability)
print('신뢰도:', confidence, '분기:', route)
print('기본 예측:', base, '최종 예측:', final)
assert np.array_equal(final, [0, 2])

확률: [[0.7  0.2  0.1 ]
 [0.4  0.35 0.25]]
신뢰도: [0.7 0.4] 분기: [False  True]
기본 예측: [0 0] 최종 예측: [0 2]


In [26]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel

training = np.array([[0.1,0.2,0.3], [0.4,0.3,0.6], [0.7,0.8,0.5], [0.9,0.6,0.8]])
testing = np.array([[0.2,0.3,0.4]])
pca = PCA(n_components=2, svd_solver='full')
scaler = MinMaxScaler(feature_range=(0, np.pi))
train_angles = scaler.fit_transform(pca.fit_transform(training))
test_angles = np.clip(scaler.transform(pca.transform(testing)), 0, np.pi)
kernel = FidelityStatevectorKernel(feature_map=zz_feature_map(2, reps=2))
K_train = kernel.evaluate(train_angles)
K_test = kernel.evaluate(test_angles, train_angles)
print('학습 커널:', K_train.shape, '시험 커널:', K_test.shape)
print('대각 성분:', np.round(np.diag(K_train), 6))
print('최소 고유값:', round(float(np.linalg.eigvalsh(K_train).min()), 6))
assert np.allclose(K_train, K_train.T)
assert np.allclose(np.diag(K_train), 1.0)
assert K_test.shape == (1, 4)

학습 커널: (4, 4) 시험 커널: (1, 4)
대각 성분: [1. 1. 1. 1.]
최소 고유값: 0.130337


In [23]:
import time
from pathlib import Path
from importlib.metadata import version

import joblib
import numpy as np
import torch
from torch import nn
from threadpoolctl import threadpool_limits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.svm import SVC
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.algorithms.classifiers import QSVC
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 1. 실행 설정
threadpool_limits(limits=1)
torch.set_num_threads(1)
torch.manual_seed(7)
started = time.perf_counter()
threshold = 0.6
max_per_class = 12
q_reps = 2
print('버전:', {p: version(p) for p in (
    'qiskit', 'qiskit-machine-learning', 'torch', 'numpy', 'scikit-learn'
)})

# 2. 고정 2차원 특징과 MLP 정의
# 좌표는 교육용 무차원 특징 점수입니다. 배열 순서 A, B, C가 라벨 0, 1, 2입니다.
A = np.array([
    [1.5,1.6],[1.8,2.0],[2.1,2.3],[2.4,1.8],
    [1.6,2.4],[2.0,1.5],[2.3,2.1],[2.6,2.5],
    [1.9,2.6],[2.2,1.7],[2.5,2.2],[2.8,1.9],
    [2.4,2.7],[2.7,1.6],[3.0,2.1],[3.3,2.4],
    [2.8,2.8],[3.1,1.8],[3.4,2.2],[3.6,2.5],
    [3.2,2.7],[3.5,1.7],[3.7,2.0],[3.9,2.3],
], dtype=np.float32)
B = np.array([
    [6.5,1.6],[6.2,2.0],[5.9,2.3],[5.6,1.8],
    [6.4,2.4],[6.0,1.5],[5.7,2.1],[5.4,2.5],
    [6.1,2.6],[5.8,1.7],[5.5,2.2],[5.2,1.9],
    [5.6,2.7],[5.3,1.6],[5.0,2.1],[4.7,2.4],
    [5.2,2.8],[4.9,1.8],[4.6,2.2],[4.4,2.5],
    [4.8,2.7],[4.5,1.7],[4.3,2.0],[4.1,2.3],
], dtype=np.float32)
C = np.array([
    [3.0,1.6],[3.3,2.0],[3.6,2.3],[3.9,1.8],
    [3.2,2.4],[3.5,1.5],[3.8,2.1],[4.1,2.5],
    [3.4,2.6],[3.7,1.7],[4.0,2.2],[4.3,1.9],
    [3.6,2.7],[3.9,1.6],[4.2,2.1],[4.5,2.4],
    [3.8,2.8],[4.1,1.8],[4.4,2.2],[4.7,2.5],
    [4.0,2.7],[4.3,1.7],[4.6,2.0],[4.9,2.3],
], dtype=np.float32)
X = np.vstack([A, B, C])
y = np.repeat(np.arange(3), 24).astype(np.int64)
role = np.tile(np.arange(24) % 4, 3)
train_mask, cal_mask, test_mask = role < 2, role == 2, role == 3
assert not np.any(train_mask & cal_mask) and not np.any(cal_mask & test_mask)
input_scaler = StandardScaler()
X_train = input_scaler.fit_transform(X[train_mask]).astype(np.float32)
X_cal = input_scaler.transform(X[cal_mask]).astype(np.float32)
X_test = input_scaler.transform(X[test_mask]).astype(np.float32)
y_train, y_cal, y_test = y[train_mask], y[cal_mask], y[test_mask]
class_names = ['A', 'B', 'C']
class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU())
        self.head = nn.Linear(16, 3)
    def encode(self, x):
        return self.features(x)
    def forward(self, x):
        return self.head(self.encode(x))

model = SmallMLP()
epochs, batch_size, learning_rate = 40, 36, 0.01
base_name = 'MLP'
output_dir = Path('day38_mlp_results')


# 3. 고전 모델 학습: train 데이터만 사용합니다.
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()
losses = []
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    order = torch.randperm(len(X_train)).numpy()
    for start in range(0, len(X_train), batch_size):
        indices = order[start:start + batch_size]
        xb = torch.from_numpy(X_train[indices])
        yb = torch.from_numpy(y_train[indices])
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.detach()) * len(xb)
    losses.append(total_loss / len(X_train))
    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f'학습 {epoch + 1}/{epochs}: loss={losses[-1]:.6f}')

# 4. 특징과 신뢰도 계산
@torch.no_grad()
def infer_base(array):
    model.eval()
    features, probabilities = [], []
    for start in range(0, len(array), batch_size):
        xb = torch.from_numpy(array[start:start + batch_size])
        embedding = model.encode(xb)
        logits = model.head(embedding)
        features.append(embedding.numpy())
        probabilities.append(torch.softmax(logits, dim=1).numpy())
    features = np.vstack(features)
    probabilities = np.vstack(probabilities)
    return features, probabilities.argmax(axis=1), probabilities.max(axis=1)

cal_features, cal_pred, cal_conf = infer_base(X_cal)
# calibration에서는 정답을 사용해 보완 학습 사례를 찾을 수 있습니다.
hard = (cal_pred != y_cal) | (cal_conf < threshold)
print('학습/보완/최종시험:', len(y_train), len(y_cal), len(y_test))
print('임베딩 차원:', cal_features.shape[1])
print('보완 데이터의 하드 샘플:', int(hard.sum()))

# 5. QSVC 학습 행 선택: 클래스별 최대 개수를 제한합니다.
selected = []
if hard.any():
    for label in range(len(class_names)):
        indices = np.flatnonzero(y_cal == label)
        # 하드 샘플을 우선하고 신뢰도가 낮은 행부터 선택합니다.
        order = sorted(indices, key=lambda i: (not hard[i], cal_conf[i], i))
        difficult = [i for i in order if hard[i]]
        # 각 클래스에 최소 두 사례를 확보합니다.
        chosen = difficult[:max_per_class]
        for i in order:
            if len(chosen) >= min(2, len(indices), max_per_class):
                break
            if i not in chosen:
                chosen.append(i)
        selected.extend(chosen)
selected = np.array(sorted(selected), dtype=int)
qsvc = rbf = pca = angle_scaler = None
can_fit = len(selected) >= 2 and len(np.unique(y_cal[selected])) >= 2
if can_fit:
    # PCA와 스케일러는 QSVC에 사용할 보완 학습 행만으로 학습합니다.
    pca = PCA(n_components=2, svd_solver='full')
    q_train = pca.fit_transform(cal_features[selected])
    angle_scaler = MinMaxScaler(feature_range=(0, np.pi))
    q_train = angle_scaler.fit_transform(q_train)
    feature_map = zz_feature_map(feature_dimension=2, reps=q_reps)
    kernel = FidelityStatevectorKernel(feature_map=feature_map)
    qsvc = QSVC(quantum_kernel=kernel, C=1.0)
    qsvc.fit(q_train, y_cal[selected])
    # 동일한 행과 특징으로 고전 RBF 커널 비교 모델을 학습합니다.
    rbf = SVC(C=1.0, kernel='rbf', gamma='scale')
    rbf.fit(q_train, y_cal[selected])
    K = kernel.evaluate(q_train)
    assert np.allclose(K, K.T, atol=1e-8)
    assert np.allclose(np.diag(K), 1.0, atol=1e-8)
    assert np.linalg.eigvalsh(K).min() >= -1e-8
    print('QSVC 학습 건수:', len(selected))
    print('QSVC 클래스별 건수:', np.bincount(y_cal[selected], minlength=len(class_names)))
    print('PCA 설명분산 합:', round(float(pca.explained_variance_ratio_.sum()), 6))
else:
    print('QSVC 학습 조건 미충족: 고전 모델 예측을 유지합니다.')

# 6. 최종 추론: 정답 없이 신뢰도만으로 분기합니다.
def hybrid_predict(array):
    features, base, confidence = infer_base(array)
    quantum, classical = base.copy(), base.copy()
    route = confidence < threshold
    used = route & (qsvc is not None)
    clipped = 0
    if used.any():
        raw = angle_scaler.transform(pca.transform(features[used]))
        clipped = int(((raw < 0) | (raw > np.pi)).sum())
        angles = np.clip(raw, 0, np.pi)
        quantum[used] = qsvc.predict(angles)
        classical[used] = rbf.predict(angles)
    return base, quantum, classical, route, used, clipped

base, quantum, classical, route, used, clipped = hybrid_predict(X_test)
# 정답은 아래의 최종 평가에서 사용합니다.
print('최종시험 중 낮은 신뢰도:', int(route.sum()))
print('실제 QSVC 호출 건수:', int(used.sum()))
print('각도 클리핑 원소 수:', clipped)
for name, prediction in [(base_name, base), ('Hybrid QSVC', quantum), ('Hybrid RBF', classical)]:
    print(name, '정확도:', round(accuracy_score(y_test, prediction), 6),
          'macro F1:', round(f1_score(y_test, prediction, average='macro', zero_division=0), 6))
print('QSVC 혼동행렬: 행=정답, 열=예측, 클래스 순서=', class_names)
print(confusion_matrix(y_test, quantum, labels=np.arange(len(class_names))))
fixed = int(((base != y_test) & (quantum == y_test)).sum())
broken = int(((base == y_test) & (quantum != y_test)).sum())
print('QSVC가 수정한 오분류:', fixed, '새로 발생한 오분류:', broken)
assert np.array_equal(quantum[~used], base[~used])
assert np.isclose((quantum == y_test).sum() - (base == y_test).sum(), fixed - broken)

# 7. 모델 저장과 재로드 후 예측 확인
output_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), output_dir / 'base_model.pt')
bundle = {
    'pca': pca, 'angle_scaler': angle_scaler,
    'qsvc': qsvc, 'rbf': rbf, 'threshold': threshold,
    'class_names': class_names, 'input_scaler': input_scaler,
    'selected_rows': selected,
}
joblib.dump(bundle, output_dir / 'hybrid_bundle.joblib')
model.load_state_dict(torch.load(output_dir / 'base_model.pt', map_location='cpu', weights_only=True))
loaded = joblib.load(output_dir / 'hybrid_bundle.joblib')
pca, angle_scaler = loaded['pca'], loaded['angle_scaler']
qsvc, rbf = loaded['qsvc'], loaded['rbf']
threshold = loaded['threshold']
reloaded = hybrid_predict(X_test)[1]
assert np.array_equal(quantum, reloaded)
print('재로드 예측 일치:', True)

# 8. 학습 곡선과 한 건 추론
fig, ax = plt.subplots(figsize=(6, 3.5), facecolor='white')
ax.plot(np.arange(1, epochs + 1), losses, color='#e87524')
ax.set(xlabel='Epoch', ylabel='Train loss')
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(output_dir / 'training_curve.png', dpi=150)
plt.close(fig)
one = hybrid_predict(X_test[:1])
print('첫 시험 항목:', '기본=', class_names[int(one[0][0])],
      '최종=', class_names[int(one[1][0])], 'QSVC 사용=', bool(one[4][0]))
print('출력 폴더:', output_dir.resolve())
print('실행 시간(초):', round(time.perf_counter() - started, 2))

버전: {'qiskit': '2.2.3', 'qiskit-machine-learning': '0.9.0', 'torch': '2.14.1+cpu', 'numpy': '2.4.6', 'scikit-learn': '1.6.1'}
학습 1/40: loss=1.098537
학습 5/40: loss=0.996809
학습 10/40: loss=0.847890
학습 15/40: loss=0.692141
학습 20/40: loss=0.558864
학습 25/40: loss=0.455063
학습 30/40: loss=0.365335
학습 35/40: loss=0.295669
학습 40/40: loss=0.245588
학습/보완/최종시험: 36 18 18
임베딩 차원: 16
보완 데이터의 하드 샘플: 5
QSVC 학습 건수: 6
QSVC 클래스별 건수: [2 2 2]
PCA 설명분산 합: 0.991984
최종시험 중 낮은 신뢰도: 2
실제 QSVC 호출 건수: 2
각도 클리핑 원소 수: 2
MLP 정확도: 0.611111 macro F1: 0.625641
Hybrid QSVC 정확도: 0.666667 macro F1: 0.671429
Hybrid RBF 정확도: 0.611111 macro F1: 0.625641
QSVC 혼동행렬: 행=정답, 열=예측, 클래스 순서= ['A', 'B', 'C']
[[4 0 2]
 [0 5 1]
 [0 3 3]]
QSVC가 수정한 오분류: 1 새로 발생한 오분류: 0
재로드 예측 일치: True
첫 시험 항목: 기본= A 최종= A QSVC 사용= False
출력 폴더: /content/day38_mlp_results
실행 시간(초): 26.98


아래 출력은 Qiskit 2.2.3, qiskit-machine-learning 0.9.0, PyTorch 2.5.1 CPU, NumPy 2.2.6 환경의 예입니다. 수치 연산 환경에 따라 세부 결과가 달라질 수 있습니다.

기본 MLP는 시험 18건 중 11건, QSVC 결합 모델은 12건을 맞혔습니다. 시험 입력 중 2건이 QSVC로 분기되었고 기존 오분류 1건이 수정됐습니다. 새로 발생한 오분류는 0건입니다. 정확도는 약 0.6111과 0.6667이며 차이는 정답 한 건에 해당합니다.

이 결과는 지정된 고정 데이터와 설정의 관측값입니다. 클래스별 오류와 macro F1, 보완 호출 건수까지 확인해야 어떤 입력에서 변화가 있었는지 알 수 있습니다. 표본이 작으므로 다른 데이터 분리와 별도 데이터에서도 평가합니다.

학습 손실 그래프는 출력 폴더의 `training_curve.png`에서 확인합니다. 손실 감소는 train 데이터에 대한 결과이며 최종 시험 성능은 별도로 해석합니다. `base_model.pt`는 고전 모델 가중치, `hybrid_bundle.joblib`은 보완 모델과 전처리·클래스·임계값을 포함합니다. 직접 생성한 파일을 같은 패키지 환경에서 읽습니다.